# Trading bot: score news with fine-tuned Laya checkpoints (out-of-sample checks)

Models come from the `trading-bot-laya-finetune` notebook's output; labels and scripts from the private dataset
`trading-bot-laya-finetune-data`. Runs `scripts/predict_laya.py` once per line of `eval.txt`
(`<name> <model dir in the finetune output> <predict args>`) and writes `/kaggle/working/<name>.jsonl`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "laya==0.3.10" python-dotenv

In [ ]:
import glob, os, shutil, subprocess, shlex
files = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*', recursive=True)
         if os.path.isfile(p) and '/trading-bot-laya-finetune-data/' in p}
repo = '/kaggle/working/repo'
for d in ('advisor', 'scripts', 'data'):
    os.makedirs(f'{repo}/{d}', exist_ok=True)
shutil.copy(files['config.py'], repo)
shutil.copy(files['laya_advisor.py'], f'{repo}/advisor/')
open(f'{repo}/advisor/__init__.py', 'w').close()
for f in ('finetune_laya.py', 'predict_laya.py'):
    shutil.copy(files[f], f'{repo}/scripts/')
for f in files:
    if f.endswith('.jsonl'):
        shutil.copy(files[f], f'{repo}/data/')
models = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/model.safetensors', recursive=True)})
print('models:', models)
jobs = [l.split(None, 2) for l in open(files['eval.txt']) if l.strip() and not l.startswith('#')]
print(jobs)

In [ ]:
for name, model, extra in jobs:
    path = next(m for m in models if m.endswith('/' + model))
    cmd = f'python scripts/predict_laya.py --model {path} --out /kaggle/working/{name}.jsonl {extra.strip()}'
    print('=' * 20, name, '|', cmd, flush=True)
    p = subprocess.run(shlex.split(cmd), cwd=repo, capture_output=True, text=True)
    print('\n'.join(l for l in (p.stdout + p.stderr).splitlines() if 'warn' not in l.lower())[-3000:], flush=True)
shutil.rmtree(repo, ignore_errors=True)